# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anshikaag-28/Anshika-FlyRank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Add it in Colab Secrets.")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("Hugging Face authentication is ready.")

Hugging Face authentication is ready.


In [3]:
query = """
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(ga4_sessions) AS sessions,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS ctr,

    AVG(gsc_avg_position) AS avg_position

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE ga4_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
"""

features = con.sql(query).df()

print("Rows:", len(features))
display(features.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 90489


,client_hash_id,content_hash_id,impressions,clicks,sessions,ctr,avg_position
0,client_9958f0a7ae1df715,content_810cf06597918291,257.0,1.0,42.0,0.389105,11.186123
1,client_9958f0a7ae1df715,content_b813c73d7000b3b1,180.0,1.0,7.0,0.555556,8.674734
2,client_9958f0a7ae1df715,content_5a77dbf5671c5a65,19657.0,199.0,168.0,1.012362,4.532382
3,client_9958f0a7ae1df715,content_f5e11209b398d173,47.0,0.0,11.0,0.000000,10.045000
4,client_9958f0a7ae1df715,content_8f3fa2db89105948,1.0,0.0,2.0,0.000000,7.000000


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I will give higher priority to pages that have sufficient search visibility but achieve a lower-than-expected CTR compared with other pages in the same position range.

The baseline score will consider both the CTR shortfall and the number of search impressions. Therefore, pages with a larger CTR opportunity and greater search exposure will receive a higher priority.

**Reason code:** `CTR_POSITION_OPPORTUNITY`

**Action:** `REFRESH`

This rule is intended only to rank pages for human review. It should not be interpreted as evidence that refreshing a page will necessarily increase its future traffic or search performance.



In [4]:
# Reuse the feature dataframe from the signal audit logic.

baseline = features.copy()

# Create position buckets
baseline["position_bucket"] = pd.cut(
    baseline["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["Top 3", "Page 1", "Page 2", "Deep"]
)

# Calculate the typical CTR for each position bucket
position_ctr = (
    baseline
    .groupby("position_bucket", observed=True)["ctr"]
    .median()
)

baseline["expected_ctr"] = baseline["position_bucket"].map(
    position_ctr
)

# CTR gap:
# positive = page CTR is below its position-bucket benchmark
# Make sure both CTR columns are numeric
baseline["expected_ctr"] = pd.to_numeric(
    baseline["expected_ctr"],
    errors="coerce"
)

baseline["ctr"] = pd.to_numeric(
    baseline["ctr"],
    errors="coerce"
)

# Positive gap = observed CTR is below the position benchmark
baseline["ctr_gap"] = (
    baseline["expected_ctr"] - baseline["ctr"]
)

# Only positive opportunities contribute to the score
baseline["score"] = (
    baseline["ctr_gap"].clip(lower=0)
    * np.log1p(baseline["impressions"])
)

print(baseline[
    ["expected_ctr", "ctr", "ctr_gap", "impressions", "score"]
].head(10))

# Keep only positive opportunities
baseline["positive_ctr_gap"] = baseline["ctr_gap"].clip(lower=0)

# Use log1p to reduce the dominance of extremely large impression counts
baseline["volume_weight"] = np.log1p(
    baseline["impressions"].clip(lower=0)
)

# Final transparent baseline score
baseline["score"] = (
    baseline["positive_ctr_gap"]
    * baseline["volume_weight"]
)

baseline["reason_code"] = np.where(
    baseline["score"] > 0,
    "CTR_POSITION_OPPORTUNITY",
    "NO_OPPORTUNITY"
)

baseline["action"] = np.where(
    baseline["score"] > 0,
    "REFRESH",
    "MONITOR"
)

print("Baseline rule created.")
display(
    baseline[
        [
            "content_hash_id",
            "impressions",
            "ctr",
            "avg_position",
            "expected_ctr",
            "ctr_gap",
            "score",
            "reason_code",
            "action"
        ]
    ].head(10)
)

   expected_ctr       ctr   ctr_gap  impressions     score
0      0.381492  0.389105 -0.007614        257.0  0.000000
1      0.558659  0.555556  0.003104        180.0  0.016134
2      0.558659  1.012362 -0.453703      19657.0  0.000000
3      0.381492  0.000000  0.381492         47.0  1.476830
4      0.558659  0.000000  0.558659          1.0  0.387233
5      0.558659  2.194357 -1.635698        319.0  0.000000
6      0.558659  0.638716 -0.080057       6106.0  0.000000
7      0.000000  1.010101 -1.010101        396.0  0.000000
8      0.381492  0.467290 -0.085798        428.0  0.000000
9      0.558659  3.759398 -3.200739        133.0  0.000000
Baseline rule created.


,content_hash_id,impressions,ctr,avg_position,expected_ctr,ctr_gap,score,reason_code,action
0,content_810cf06597918291,257.0,0.389105,11.186123,0.381492,-0.007614,0.000000,NO_OPPORTUNITY,MONITOR
1,content_b813c73d7000b3b1,180.0,0.555556,8.674734,0.558659,0.003104,0.016134,CTR_POSITION_OPPORTUNITY,REFRESH
2,content_5a77dbf5671c5a65,19657.0,1.012362,4.532382,0.558659,-0.453703,0.000000,NO_OPPORTUNITY,MONITOR
3,content_f5e11209b398d173,47.0,0.000000,10.045000,0.381492,0.381492,1.476830,CTR_POSITION_OPPORTUNITY,REFRESH
4,content_8f3fa2db89105948,1.0,0.000000,7.000000,0.558659,0.558659,0.387233,CTR_POSITION_OPPORTUNITY,REFRESH
5,content_278030b007943b07,319.0,2.194357,5.914484,0.558659,-1.635698,0.000000,NO_OPPORTUNITY,MONITOR
6,content_237e63fc00c8c7ad,6106.0,0.638716,7.108246,0.558659,-0.080057,0.000000,NO_OPPORTUNITY,MONITOR
7,content_347fbafb77d3ae37,396.0,1.010101,21.671343,0.000000,-1.010101,0.000000,NO_OPPORTUNITY,MONITOR
8,content_15bd72d24e0a0b08,428.0,0.467290,13.669138,0.381492,-0.085798,0.000000,NO_OPPORTUNITY,MONITOR
9,content_6f4cc70af7be346e,133.0,3.759398,9.234436,0.558659,-3.200739,0.000000,NO_OPPORTUNITY,MONITOR


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# Rank all pages by baseline score
import os

queue = (
    baseline
    .sort_values(
        by=["score", "impressions"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "impressions",
        "clicks",
        "sessions",
        "ctr",
        "avg_position",
        "expected_ctr",
        "ctr_gap"
    ]
]

print("Ranked queue:")
display(queue.head(20))

print("\nTotal rows:", len(queue))

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

queue.to_csv(
    output_path,
    index=False
)

print("\nCSV written successfully:")
print(output_path)

print("File exists:", os.path.exists(output_path))

Ranked queue:


,rank,client_hash_id,content_hash_id,score,reason_code,action,impressions,clicks,sessions,ctr,avg_position,expected_ctr,ctr_gap
0,1,client_23a62021009f63c4,content_44f34c0a90047651,6.614834,CTR_POSITION_OPPORTUNITY,REFRESH,168160.0,15.0,37.0,0.008920,7.324954,0.558659,0.549739
1,2,client_e547b89c05043229,content_306bc78dff1eb683,6.176824,CTR_POSITION_OPPORTUNITY,REFRESH,42288.0,25.0,22.0,0.059118,1.427150,0.638978,0.579859
2,3,client_e547b89c05043229,content_9ef3d7516483e665,6.011176,CTR_POSITION_OPPORTUNITY,REFRESH,79852.0,85.0,42.0,0.106447,2.504120,0.638978,0.532531
3,4,client_e547b89c05043229,content_b2b85c287474668d,5.962808,CTR_POSITION_OPPORTUNITY,REFRESH,51479.0,46.0,41.0,0.089357,1.471562,0.638978,0.549621
4,5,client_23a62021009f63c4,content_bf078007df823490,5.880311,CTR_POSITION_OPPORTUNITY,REFRESH,37262.0,0.0,12.0,0.000000,7.518648,0.558659,0.558659
5,6,client_e547b89c05043229,content_c46df0fa61530d86,5.873374,CTR_POSITION_OPPORTUNITY,REFRESH,37877.0,31.0,39.0,0.081844,0.920858,0.638978,0.557134
6,7,client_e547b89c05043229,content_8d7d99f109e19aa2,5.835111,CTR_POSITION_OPPORTUNITY,REFRESH,181942.0,286.0,164.0,0.157193,2.568135,0.638978,0.481785
7,8,client_e5c2aa26a8598242,content_4977e90c4d93cf9f,5.833682,CTR_POSITION_OPPORTUNITY,REFRESH,59927.0,17.0,86.0,0.028368,7.308634,0.558659,0.530291
8,9,client_73cda7b4e4f265ea,content_252aa5480bb1f8d7,5.661188,CTR_POSITION_OPPORTUNITY,REFRESH,26477.0,22.0,23.0,0.083091,2.117813,0.638978,0.555887
9,10,client_23a62021009f63c4,content_fe8baba849843607,5.641117,CTR_POSITION_OPPORTUNITY,REFRESH,6824.0,0.0,115.0,0.000000,2.749461,0.638978,0.638978



Total rows: 90489

CSV written successfully:
work/outputs/baseline_action_score.csv
File exists: True


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

A page receiving a high baseline score does not necessarily mean that it should be refreshed immediately. The score only indicates that the page matches the conditions defined by the rule. The recommendation should therefore be reviewed by considering what factors could make the rule inaccurate.

Some possible issues are:

* **Low volume:** A page with very few impressions may have an unreliable CTR. A high or low observed CTR could simply be due to limited data.

* **Position problem:** The average search position may not fully represent the different queries and positions associated with the page, which can make the comparison less accurate.

* **Seasonal demand:** The page's performance may be affected by seasonal changes in user interest. In that case, the observed pattern may not indicate a need for content improvement.

* **Search-result changes:** A change in the search-results layout or other SERP behaviour could affect the number of clicks without any change in the actual quality of the page.

Therefore, the purpose of reviewing a high-scoring page is to question the recommendation and identify circumstances where the baseline rule may not be reliable.


In [6]:
top20 = queue.head(20).copy()

top20["confidence_note"] = np.where(
    top20["impressions"] >= 500,
    "More search exposure; still needs human review.",
    "Lower volume; recommendation may be noisy."
)

top20["what_would_make_it_wrong"] = np.where(
    top20["impressions"] < 500,
    "Low volume could make the CTR gap unstable.",
    "Position estimate or CTR may not represent a persistent opportunity."
)

display(
    top20[
        [
            "rank",
            "action",
            "reason_code",
            "score",
            "impressions",
            "ctr",
            "avg_position",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,action,reason_code,score,impressions,ctr,avg_position,confidence_note,what_would_make_it_wrong
0,1,REFRESH,CTR_POSITION_OPPORTUNITY,6.614834,168160.0,0.008920,7.324954,More search exposure; still needs human review.,Position estimate or CTR may not represent a p...
1,2,REFRESH,CTR_POSITION_OPPORTUNITY,6.176824,42288.0,0.059118,1.427150,More search exposure; still needs human review.,Position estimate or CTR may not represent a p...
2,3,REFRESH,CTR_POSITION_OPPORTUNITY,6.011176,79852.0,0.106447,2.504120,More search exposure; still needs human review.,Position estimate or CTR may not represent a p...
3,4,REFRESH,CTR_POSITION_OPPORTUNITY,5.962808,51479.0,0.089357,1.471562,More search exposure; still needs human review.,Position estimate or CTR may not represent a p...
4,5,REFRESH,CTR_POSITION_OPPORTUNITY,5.880311,37262.0,0.000000,7.518648,More search exposure; still needs human review.,Position estimate or CTR may not represent a p...
5,6,REFRESH,CTR_POSITION_OPPORTUNITY,5.873374,37877.0,0.081844,0.920858,More search exposure; still needs human review.,Position estimate or CTR may not represent a p...
6,7,REFRESH,CTR_POSITION_OPPORTUNITY,5.835111,181942.0,0.157193,2.568135,More search exposure; still needs human review.,Position estimate or CTR may not represent a p...
7,8,REFRESH,CTR_POSITION_OPPORTUNITY,5.833682,59927.0,0.028368,7.308634,More search exposure; still needs human review.,Position estimate or CTR may not represent a p...
8,9,REFRESH,CTR_POSITION_OPPORTUNITY,5.661188,26477.0,0.083091,2.117813,More search exposure; still needs human review.,Position estimate or CTR may not represent a p...
9,10,REFRESH,CTR_POSITION_OPPORTUNITY,5.641117,6824.0,0.000000,2.749461,More search exposure; still needs human review.,Position estimate or CTR may not represent a p...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
# Inspect the bottom of the ranked opportunity list

weak_picks = queue.tail(10)

print("Weak / low-score picks:")
display(
    weak_picks[
        [
            "rank",
            "score",
            "reason_code",
            "action",
            "impressions",
            "ctr",
            "avg_position"
        ]
    ]
)

# Leakage check
forbidden_columns = {
    "label",
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "future_clicks",
    "future_sessions",
    "future_impressions"
}

used_columns = set(baseline.columns)

leaked_columns = forbidden_columns.intersection(
    used_columns
)

print("\nPotential label/future columns found:")
print(leaked_columns)

assert len(leaked_columns) == 0

print("\nPASS: No listed label-derived or future-window fields are used.")

Weak / low-score picks:


,rank,score,reason_code,action,impressions,ctr,avg_position
90479,90480,NaN,NO_OPPORTUNITY,MONITOR,0.0,NaN,NaN
90480,90481,NaN,NO_OPPORTUNITY,MONITOR,0.0,NaN,NaN
90481,90482,NaN,NO_OPPORTUNITY,MONITOR,0.0,NaN,NaN
90482,90483,NaN,NO_OPPORTUNITY,MONITOR,0.0,NaN,NaN
90483,90484,NaN,NO_OPPORTUNITY,MONITOR,0.0,NaN,NaN
90484,90485,NaN,NO_OPPORTUNITY,MONITOR,0.0,NaN,NaN
90485,90486,NaN,NO_OPPORTUNITY,MONITOR,0.0,NaN,NaN
90486,90487,NaN,NO_OPPORTUNITY,MONITOR,0.0,NaN,NaN
90487,90488,NaN,NO_OPPORTUNITY,MONITOR,0.0,NaN,NaN
90488,90489,NaN,NO_OPPORTUNITY,MONITOR,0.0,NaN,NaN



Potential label/future columns found:
set()

PASS: No listed label-derived or future-window fields are used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.